In [1]:
# Classes and funcctions

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate

from qiskit_aer import AerSimulator
import math

     

In [2]:
# Hadamard gate rotates the state of a qubit to an equal superposition of |0⟩ and |1⟩, which is essential for generating random bits.

simulator = AerSimulator()

def quantum_random_bit():
    """Returns a random 0 or 1 by measuring a qubit in superposition."""
    qc = QuantumCircuit(1, 1)   # 1 qubit, 1 classical bit
    qc.h(0)                      # Hadamard gate: |0⟩ → (|0⟩+|1⟩)/√2
    qc.measure(0, 0)             # measure qubit 0 → store in classical bit 0
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts())[0])

def quantum_random_bits(n):
    """Returns a list of n random bits."""
    return [quantum_random_bit() for _ in range(n)]

In [3]:

# Using 2 bases to encode and measure qubits

def encode_qubit(bit, basis):
    """
    Alice encodes one classical bit into a qubit.
      bit   = 0 or 1 (the secret value)
      basis = 0 → Z-basis (rectilinear): uses |0⟩ or |1⟩
              1 → X-basis (diagonal):    uses |+⟩ or |−⟩
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)    # X gate flips |0⟩ → |1⟩
    if basis == 1:
        qc.h(0)    # H gate rotates to diagonal basis: |0⟩→|+⟩, |1⟩→|−⟩
    return qc      # return the circuit (the "qubit" to send to Bob)

def measure_qubit(qc, basis):
    """
    Bob measures a qubit in his chosen basis.
      basis = 0 → measure in Z-basis (no extra gate needed)
              1 → measure in X-basis (apply H first to rotate back)
    """
    qc = qc.copy()    # not modifying Alice's original circuit
    if basis == 1:
        qc.h(0)       # rotate from X-basis back to Z before measuring
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts())[0])

In [5]:
N_BITS     = 100   # number of qubits Alice sends to Bob
CHECK_FRAC = 0.5   # fraction of sifted key for error checking
THRESHOLD  = 0.11  # declare the attack if error rate exceeds this threshold

In [6]:
# === ALICE ===

# Alice independently generates two random bit strings of length N_BITS:
alice_bits  = quantum_random_bits(N_BITS)   # secret bit values
alice_bases = quantum_random_bits(N_BITS)   # random basis for each qubit

# Encode each bit into a qubit using Alice's chosen basis
qubits = [encode_qubit(b, basis)
          for b, basis in zip(alice_bits, alice_bases)]

print(f"Alice bits  (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]}")

Alice bits  (first 20): [1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0]
Alice bases (first 20): [1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]


In [7]:
# === BOB ===

# Bob would not be able to guess Alice's random bits or bases, so he also generates his own random basis choices for each qubit.
bob_bases   = quantum_random_bits(N_BITS)    # Bob's random basis choices

# Bob measures each qubit using his own randomly chosen basis
bob_results = [measure_qubit(qubits[i], bob_bases[i])
               for i in range(N_BITS)]

print(f"Bob bases   (first 20): {bob_bases[:20]}")
print(f"Bob results (first 20): {bob_results[:20]}")

Bob bases   (first 20): [0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
Bob results (first 20): [0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0]


In [8]:
# === ALICE + BOB (classical public channel) ===

# Alice and Bob now talk over a classical public channel, broadcast their basis choices. 

# Find positions where they happened to use the same basis
matching  = [i for i in range(N_BITS) if alice_bases[i] == bob_bases[i]]

# Keep only those bit values — this is the "sifted key"
alice_key = [alice_bits[i]  for i in matching]
bob_key   = [bob_results[i] for i in matching]

print(f"Sifted key length : {len(alice_key)} bits  (~50% of {N_BITS} expected)")
print(f"Alice sifted (first 20): {alice_key[:20]}")
print(f"Bob   sifted (first 20): {bob_key[:20]}")

Sifted key length : 50 bits  (~50% of 100 expected)
Alice sifted (first 20): [1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0]
Bob   sifted (first 20): [1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0]


In [9]:
# === ALICE + BOB ===

# Select a random subset of the sifted key to check publicly
n_check   = math.ceil(len(alice_key) * CHECK_FRAC)
check_idx = sorted(set(quantum_random_bits(n_check * 3)))[:n_check]

# Compare Alice's and Bob's values at those positions
alice_check = [alice_key[i] for i in check_idx]
bob_check   = [bob_key[i]   for i in check_idx]

errors = sum(a != b for a, b in zip(alice_check, bob_check))
qber   = errors / len(check_idx)    # Quantum Bit Error Rate

# The remaining positions become the final secret key
final_idx = [i for i in range(len(alice_key)) if i not in check_idx]
final_key = [alice_key[i] for i in final_idx]

print(f"Bits checked : {len(check_idx)}")
print(f"Errors found : {errors}")
print(f"QBER         : {qber:.1%}")
print()

# No attacker -> QBER = 0% -> key accepted
# Eve present -> QBER = 25% -> attack detected and key discarded

if qber > THRESHOLD:
    print("ATTACK DETECTED! QBER exceeds threshold. Discard key.")
else:
    print("Channel secure. Shared key established.")
    print(f"Final key ({len(final_key)} bits): {final_key[:30]}...")

Bits checked : 2
Errors found : 0
QBER         : 0.0%

Channel secure. Shared key established.
Final key (48 bits): [0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1]...
